# Section 4 · Notebook 1 — Supervisor–Worker Pattern

A market-research assistant:
- **News Agent** — recent news and developments (Tavily web search)
- **Financial Agent** — prices, history and fundamentals (Yahoo Finance via `yfinance`)
- **Sentiment Agent** — what investors are saying (Tavily web search)
- **Supervisor** — splits the question, delegates to the workers, and writes one recommendation

```
                    ┌──────────────┐
   question ──────► │  Supervisor  │ ──────► one combined answer
                    └──────┬───────┘
          ┌────────────────┼────────────────┐
          ▼                ▼                ▼
     News Agent     Financial Agent   Sentiment Agent
```

**You need** a free [Tavily](https://www.tavily.com/) API key. An [Alpha Vantage](https://www.alphavantage.co/support/#api-key) key is optional.

In [1]:
%pip install -q -r requirements.txt


[notice] A new release of pip is available: 24.3.1 -> 26.3
[notice] To update, run: /private/tmp/scaler-code-venv/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
from strands.models.bedrock import BedrockModel

MODEL_ID = os.getenv("BEDROCK_MODEL_ID", "amazon.nova-lite-v1:0")
REGION = os.getenv("AWS_REGION", os.getenv("AWS_DEFAULT_REGION", "us-east-1"))
os.environ.setdefault("AWS_REGION", REGION)

model = BedrockModel(model_id=MODEL_ID, region_name=REGION)
print(f"Using {MODEL_ID} in {REGION}")

Using amazon.nova-lite-v1:0 in us-east-1


## Configure API keys

Put them in a `.env` file at the repo root (recommended — it is git-ignored), or set them in your shell before starting Jupyter:

```bash
export TAVILY_API_KEY="tvly-..."
export ALPHA_VANTAGE_API_KEY="..."   # optional
```

…or uncomment the lines below.

In [2]:
from dotenv import load_dotenv

load_dotenv()  # reads TAVILY_API_KEY (and ALPHA_VANTAGE_API_KEY) from a .env file, if present
# os.environ["TAVILY_API_KEY"] = "tvly-..."
# os.environ["ALPHA_VANTAGE_API_KEY"] = "..."

if not os.getenv("TAVILY_API_KEY"):
    print("⚠️  TAVILY_API_KEY not set — the news and sentiment agents cannot search the web.")
else:
    print("✓ TAVILY_API_KEY is configured")

if not os.getenv("ALPHA_VANTAGE_API_KEY"):
    print("ℹ️  ALPHA_VANTAGE_API_KEY not set — financial data comes from yfinance (free, no key).")
else:
    print("✓ ALPHA_VANTAGE_API_KEY is configured")

✓ TAVILY_API_KEY is configured
✓ ALPHA_VANTAGE_API_KEY is configured


In [3]:
import yfinance as yf
from strands import Agent, tool
from strands_tools.tavily import tavily_search

## Custom financial tools (Yahoo Finance)

In [4]:
def _num(value, money=False, pct=False):
    """Format numbers safely — Yahoo sometimes leaves fields empty."""
    if not isinstance(value, (int, float)):
        return "N/A"
    if pct:
        return f"{value * 100:.2f}%"
    return f"${value:,.2f}" if money else f"{value:,.2f}"


@tool
def get_stock_price(symbol: str) -> str:
    """Get the current stock price and key metrics using Yahoo Finance.

    Args:
        symbol: Stock ticker symbol (e.g., 'AAPL', 'TSLA', 'MSFT')
    """
    try:
        info = yf.Ticker(symbol).info
        return (
            f"Stock: {symbol}\n"
            f"Current Price: {_num(info.get('currentPrice') or info.get('regularMarketPrice'), money=True)}\n"
            f"Market Cap: {_num(info.get('marketCap'), money=True)}\n"
            f"P/E Ratio: {_num(info.get('trailingPE'))}\n"
            f"52 Week High: {_num(info.get('fiftyTwoWeekHigh'), money=True)}\n"
            f"52 Week Low: {_num(info.get('fiftyTwoWeekLow'), money=True)}\n"
            f"Average Volume: {_num(info.get('averageVolume'))}\n"
            f"Sector: {info.get('sector', 'N/A')}\n"
            f"Industry: {info.get('industry', 'N/A')}"
        )
    except Exception as e:
        return f"Error fetching stock data for {symbol}: {e}"


@tool
def get_stock_history(symbol: str, period: str = "1mo") -> str:
    """Get historical stock price data.

    Args:
        symbol: Stock ticker symbol (e.g., 'AAPL', 'TSLA')
        period: One of 1d, 5d, 1mo, 3mo, 6mo, 1y, 2y, 5y, 10y, ytd, max
    """
    try:
        hist = yf.Ticker(symbol).history(period=period)
        if hist.empty:
            return f"No historical data available for {symbol}"
        start_price = hist["Close"].iloc[0]
        end_price = hist["Close"].iloc[-1]
        change_pct = (end_price - start_price) / start_price * 100
        return (
            f"Historical Data for {symbol} ({period}):\n"
            f"Starting Price: ${start_price:.2f}\n"
            f"Ending Price: ${end_price:.2f}\n"
            f"Change: {change_pct:+.2f}%\n"
            f"Highest: ${hist['High'].max():.2f}\n"
            f"Lowest: ${hist['Low'].min():.2f}\n"
            f"Average Volume: {hist['Volume'].mean():,.0f}\n\n"
            f"Recent 5-day trend:\n{hist[['Close', 'Volume']].tail().to_string()}"
        )
    except Exception as e:
        return f"Error fetching historical data for {symbol}: {e}"


@tool
def get_company_financials(symbol: str) -> str:
    """Get company financial metrics: revenue, margins, growth and leverage.

    Args:
        symbol: Stock ticker symbol (e.g., 'AAPL', 'TSLA')
    """
    try:
        info = yf.Ticker(symbol).info
        return (
            f"Financial Metrics for {symbol}:\n"
            f"Revenue: {_num(info.get('totalRevenue'), money=True)}\n"
            f"Revenue Growth: {_num(info.get('revenueGrowth'), pct=True)}\n"
            f"Gross Profit: {_num(info.get('grossProfits'), money=True)}\n"
            f"Profit Margin: {_num(info.get('profitMargins'), pct=True)}\n"
            f"Operating Margin: {_num(info.get('operatingMargins'), pct=True)}\n"
            f"EBITDA: {_num(info.get('ebitda'), money=True)}\n"
            f"Earnings Growth: {_num(info.get('earningsGrowth'), pct=True)}\n"
            f"Return on Equity: {_num(info.get('returnOnEquity'), pct=True)}\n"
            f"Debt to Equity: {_num(info.get('debtToEquity'))}\n"
            f"Free Cash Flow: {_num(info.get('freeCashflow'), money=True)}"
        )
    except Exception as e:
        return f"Error fetching financial data for {symbol}: {e}"


print(get_stock_price(symbol="TITAN.NS"))  # tools are still plain functions you can call

Stock: TITAN.NS
Current Price: $5,005.00
Market Cap: $4,440,998,281,216.00
P/E Ratio: 63.35
52 Week High: $5,186.70
52 Week Low: $3,751.00
Average Volume: 802,365.00
Sector: Consumer Cyclical
Industry: Luxury Goods


## Optional: Alpha Vantage tool

Use this instead of (or alongside) yfinance if you have an Alpha Vantage key. Other providers worth knowing: Polygon.io, Finnhub, Financial Modeling Prep.

In [5]:
from alpha_vantage.timeseries import TimeSeries


@tool
def get_alpha_vantage_quote(symbol: str) -> str:
    """Get the latest stock quote from the Alpha Vantage API.

    Args:
        symbol: Stock ticker symbol (e.g., 'AAPL')
    """
    key = os.getenv("ALPHA_VANTAGE_API_KEY")
    if not key:
        return "ALPHA_VANTAGE_API_KEY is not set."
    ts = TimeSeries(key=key, output_format="pandas")
    data, _ = ts.get_quote_endpoint(symbol=symbol)
    return data.to_string()


financial_tools = [get_stock_price, get_stock_history, get_company_financials, tavily_search]
if os.getenv("ALPHA_VANTAGE_API_KEY"):
    financial_tools.append(get_alpha_vantage_quote)

## Worker agents

In [6]:
news_agent = Agent(
    name="news_agent",
    description="Researches recent news and developments about companies and industries.",
    model=model,
    system_prompt="""You are a news research specialist. Search for recent news and developments about companies and industries.

When searching:
- Focus on recent news (use time_range='week' or 'month')
- Prefer reliable sources such as major news outlets
- Summarize key developments and trends
- Include specific facts, dates, and figures when available""",
    tools=[tavily_search],
    callback_handler=None,
)

financial_agent = Agent(
    name="financial_agent",
    description="Analyzes stock prices, price history, and company financial metrics.",
    model=model,
    system_prompt="""You are a financial analysis specialist. Analyze financial metrics, stock performance, and company valuations.

When analyzing:
- Use get_stock_price for current prices and key metrics
- Use get_stock_history for price trends
- Use get_company_financials for revenue, earnings, and growth
- Compare companies within the same sector and give specific numbers""",
    tools=financial_tools,
    callback_handler=None,
)

sentiment_agent = Agent(
    name="sentiment_agent",
    description="Analyzes market and investor sentiment from forums, social media, and commentary.",
    model=model,
    system_prompt="""You are a market sentiment analyst. Analyze market sentiment from social media, forums, and investor discussions.

When analyzing sentiment:
- Look for both bullish and bearish opinions
- Identify common concerns and excitement points
- Distinguish retail from institutional sentiment
- Summarize the overall market mood""",
    tools=[tavily_search],
    callback_handler=None,
)
print("✓ Worker agents created")

✓ Worker agents created


## Supervisor agent

`agent.as_tool()` wraps a worker so the supervisor can call it like any other tool. (Passing the agents directly in `tools=[...]` does the same thing.)

In [7]:
supervisor = Agent(
    name="research_supervisor",
    model=model,
    system_prompt="""You are a market research coordinator. When asked about investment opportunities:

1. Work out what information is needed
2. Delegate to the specialist agents:
   - news_agent: recent developments and industry news
   - financial_agent: financial metrics and performance data
   - sentiment_agent: market sentiment and investor opinions
3. Synthesize their responses into one balanced recommendation

This is educational analysis, not financial advice — say so briefly at the end.""",
    tools=[news_agent.as_tool(), financial_agent.as_tool(), sentiment_agent.as_tool()],
)
print("✓ Supervisor created")

✓ Supervisor created


## Run the multi-agent system

In [8]:
question = "Should I invest in Custom Synthesis (CDMO) companies?"
print(f"Question: {question}\n")

response = supervisor(question)

Question: Should I invest in Custom Synthesis (CDMO) companies?

<thinking>To answer whether to invest in Custom Synthesis (CDMO) companies, we need to gather information on recent developments, financial metrics, and market sentiment. The news_agent can provide recent news and developments, the financial_agent can analyze financial metrics, and the sentiment_agent can provide market sentiment. I will start by asking the news_agent for recent news and developments about CDMO companies.</thinking>

Tool #1: news_agent


DEPRECATION WARNING: tavily is deprecated. This warning becomes an import error in v0.9.0. Migration path: use the official Tavily MCP server, documented at https://docs.tavily.com/documentation/mcp.
DEPRECATION WARNING: tavily is deprecated. This warning becomes an import error in v0.9.0. Migration path: use the official Tavily MCP server, documented at https://docs.tavily.com/documentation/mcp.


### Educational Analysis on Investing in Custom Synthesis (CDMO) Companies

Based on the general insights and trends up until 2022, here is an analysis that can help you make an informed decision about investing in Custom Synthesis (CDMO) companies. Please note that this is educational analysis and not financial advice.

#### Key Factors to Consider

1. **Expansion and Capacity Building:**
   - Many CDMO companies have been investing in expanding their manufacturing capacities. This indicates a growing demand for their services and the potential for increased revenue. Companies like Lonza and WuXi Biologics have announced expansions of their facilities to enhance production capabilities. Look for companies that are investing in capacity building as it can be a sign of future growth.

2. **Technological Advancements:**
   - Innovations in biotechnology and manufacturing processes have driven demand for specialized CDMO services. Companies that are investing in new technologies to improv

## Try different questions

In [9]:
questions = [
    "What's the outlook for renewable energy stocks?",
    "Is the semiconductor industry a good investment right now?",
]

for q in questions:
    print(f"\n{'=' * 80}\nQuestion: {q}\n{'=' * 80}")
    supervisor(q)


Question: What's the outlook for renewable energy stocks?
<thinking>To provide an outlook for renewable energy stocks, I need to gather information on recent developments, financial metrics, and market sentiment. I will start by asking the news_agent for recent news and developments about renewable energy stocks, and then the financial_agent for financial metrics and performance data. Finally, I will ask the sentiment_agent for market sentiment and investor opinions.</thinking> 
Tool #2: news_agent


DEPRECATION WARNING: tavily is deprecated. This warning becomes an import error in v0.9.0. Migration path: use the official Tavily MCP server, documented at https://docs.tavily.com/documentation/mcp.
DEPRECATION WARNING: tavily is deprecated. This warning becomes an import error in v0.9.0. Migration path: use the official Tavily MCP server, documented at https://docs.tavily.com/documentation/mcp.
DEPRECATION WARNING: tavily is deprecated. This warning becomes an import error in v0.9.0. Migration path: use the official Tavily MCP server, documented at https://docs.tavily.com/documentation/mcp.
DEPRECATION WARNING: tavily is deprecated. This warning becomes an import error in v0.9.0. Migration path: use the official Tavily MCP server, documented at https://docs.tavily.com/documentation/mcp.


### Educational Analysis on the Outlook for Renewable Energy Stocks

Based on the general trends and insights up until 2022, here is an analysis that can help you understand the outlook for renewable energy stocks. Please note that this is educational analysis and not financial advice.

#### Key Factors to Consider

1. **Growing Demand for Renewable Energy:**
   - There is a global shift towards renewable energy sources due to environmental concerns, government policies, and technological advancements. This increasing demand for renewable energy is likely to drive growth in the renewable energy sector.

2. **Technological Advancements:**
   - Innovations in renewable energy technologies, such as solar, wind, and battery storage, have been significant. These advancements are improving efficiency, reducing costs, and expanding the potential for renewable energy adoption.

3. **Government Policies and Incentives:**
   - Many governments around the world are implementing policies and provi

DEPRECATION WARNING: tavily is deprecated. This warning becomes an import error in v0.9.0. Migration path: use the official Tavily MCP server, documented at https://docs.tavily.com/documentation/mcp.
DEPRECATION WARNING: tavily is deprecated. This warning becomes an import error in v0.9.0. Migration path: use the official Tavily MCP server, documented at https://docs.tavily.com/documentation/mcp.
DEPRECATION WARNING: tavily is deprecated. This warning becomes an import error in v0.9.0. Migration path: use the official Tavily MCP server, documented at https://docs.tavily.com/documentation/mcp.


### Educational Analysis on the Semiconductor Industry as an Investment

Based on the general trends and insights up until 2022, here is an analysis that can help you understand whether the semiconductor industry is a good investment right now. Please note that this is educational analysis and not financial advice.

#### Key Factors to Consider

1. **Chip Shortages:**
   - The global semiconductor shortage has had a significant impact on various industries, leading to increased prices and production delays. This shortage has created opportunities for companies that can increase their production capacities. Look for companies that are investing in new manufacturing capabilities to address the shortage.

2. **Investment in Manufacturing:**
   - There has been a surge in investment in semiconductor manufacturing. Companies like TSMC, Intel, and Samsung are announcing plans to expand their production capacities. This indicates a strong demand for semiconductors and potential for future gro

## Key takeaways

1. **Specialisation** — each worker has one job and the right tools for it
2. **Coordination** — the supervisor decides who to ask and combines the answers
3. **Modularity** — add a new worker without touching the others

**Make it production-ready:** retries and caching for API calls, rate limits, async execution, and tracing (Section 7).